> **Legacy branched DQN demo.** Canonical training is Path B in `kaggriculture-self-training/kaggriculture-self-training.ipynb` (hierarchical DQN + bootstrap + self-play).
>
> This notebook uses the shared `encode_observation` / `encode_action` contract and `DuelingDoubleDQNBranching` (farmer + 6 hands + market).


## Setup


In [ ]:
import glob
import json
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim

_CODE = Path.cwd()
for cand in (
    _CODE,
    _CODE / "datasets" / "scottweeden" / "self-training-code",
    Path("/kaggle/input/datasets/scottweeden/kaggriculture-self-training-code"),
    Path("/kaggle/input/kaggriculture-self-training-code"),
):
    if (cand / "kaggriculture_adapter.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break

from dataset_loader import parse_kaggriculture_episode
from kaggle_env_wrapper import KaggleEnvWrapper
from kaggriculture_rl.dqn import (
    DuelingDoubleDQNBranching,
    KaggricultureFeatureExtractor,
    ReplayBuffer,
)
from kaggriculture_rl.dqn_sb3 import DQN

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print("device:", device)


## Seed branched replay buffer from episode JSON


In [ ]:
buffer = ReplayBuffer(capacity=50_000, use_priority=False)

json_files = sorted(glob.glob("replays/*.json"))
if not json_files:
    for root in (
        Path("working/kaggle_episodes"),
        Path("/kaggle/working/kaggle_episodes"),
        Path.home() / "kagg" / "working" / "kaggle_episodes",
    ):
        found = sorted(root.rglob("*.json")) if root.exists() else []
        json_files = [str(p) for p in found if p.stem.isdigit()]
        if json_files:
            break

print(f"Found {len(json_files)} episode files")
n_loaded = 0
for path in json_files[:20]:
    with open(path, "r") as f:
        data = json.load(f)
    for tr in parse_kaggriculture_episode(data, device="cpu"):
        buffer.store(
            tr["state"],
            tr["action"],
            tr["reward"],
            tr["next_state"],
            tr["done"],
        )
        n_loaded += 1

print(f"Replay buffer size: {len(buffer)} (stored {n_loaded} transitions)")
if len(buffer) == 0:
    print("No transitions loaded — BC will be skipped; online learn() still runs.")


## Offline BC on farmer + market heads


In [ ]:
extractor = KaggricultureFeatureExtractor()
policy = DuelingDoubleDQNBranching(feature_extractor=extractor).to(device)
optimizer = optim.Adam(policy.parameters(), lr=1e-3)

bc_epochs = 50
batch_size = 64
losses = []

if len(buffer) > 0:
    policy.train()
    for epoch in range(bc_epochs):
        batch = buffer.sample(min(batch_size, len(buffer)))
        s = {
            k: torch.as_tensor(v, device=device)
            for k, v in batch["state"].items()
        }
        s["tiles"] = s["tiles"].long()
        a_farmer = torch.as_tensor(batch["action_farmer"], device=device, dtype=torch.long)
        a_market = torch.as_tensor(batch["action_market"], device=device, dtype=torch.long)
        a_hands = [
            torch.as_tensor(h, device=device, dtype=torch.long)
            for h in batch["action_hands"]
        ]

        out = policy(s)
        loss = F.cross_entropy(out["farmer_q"], a_farmer)
        loss = loss + F.cross_entropy(out["market_q"], a_market)
        for i, hand_q in enumerate(out["hand_q"]):
            loss = loss + 0.1 * F.cross_entropy(hand_q, a_hands[i])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))
        if (epoch + 1) % 10 == 0:
            print(f"BC epoch {epoch+1}/{bc_epochs} loss={losses[-1]:.4f}")
    print(f"BC done. final loss={losses[-1]:.4f}")
else:
    print("Skipped BC (empty buffer).")


## Online fine-tune with KaggleEnvWrapper + DQN.learn()


In [ ]:
env = KaggleEnvWrapper.make(device=str(device), use_masking=True)

model = DQN(
    "KaggricultureCNN",
    env,
    device=str(device),
    learning_starts=0,
    buffer_size=20_000,
    batch_size=32,
    train_freq=1,
    verbose=1,
    learning_rate=1e-4,
)

if len(buffer) > 0:
    model.online_network.load_state_dict(policy.state_dict())
    model.target_network.load_state_dict(policy.state_dict())

fine_tune_steps = 64
model.learn(total_timesteps=fine_tune_steps, log_interval=16)
print(f"Online fine-tune complete ({fine_tune_steps} steps).")
print("For full training use Path B: kaggriculture_self_play_training.train_self_play")
